In [54]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


In [55]:
df=pd.read_csv("Used_Car_Price_Prediction.csv")
display(df.head())
print(df.isnull().sum())

,car_name,yr_mfr,fuel_type,kms_run,sale_price,city,times_viewed,body_type,transmission,variant,...,total_owners,broker_quote,original_price,car_rating,ad_created_on,fitness_certificate,emi_starts_from,booking_down_pymnt,reserved,warranty_avail
0,maruti swift,2015,petrol,8063,386399,noida,18715,hatchback,manual,lxi opt,...,2,397677,404177.0,great,2021-04-04T07:09:18.583,True,8975,57960,False,False
1,maruti alto 800,2016,petrol,23104,265499,noida,2676,hatchback,manual,lxi,...,1,272935,354313.0,great,2021-03-22T14:07:32.833,True,6167,39825,False,False
2,hyundai grand i10,2017,petrol,23402,477699,noida,609,hatchback,manual,sports 1.2 vtvt,...,1,469605,NaN,great,2021-03-20T05:36:31.311,True,11096,71655,False,False
3,maruti swift,2013,diesel,39124,307999,noida,6511,hatchback,manual,vdi,...,1,294262,374326.0,great,2021-01-21T12:59:19.299,True,7154,46200,False,False
4,hyundai grand i10,2015,petrol,22116,361499,noida,3225,hatchback,manual,magna 1.2 vtvt,...,1,360716,367216.0,great,2021-04-01T13:33:40.733,True,8397,54225,False,False


car_name                  0
yr_mfr                    0
fuel_type                 0
kms_run                   0
sale_price                0
city                      0
times_viewed              0
body_type               103
transmission            556
variant                   0
assured_buy               0
registered_city          10
registered_state         10
is_hot                    0
rto                       0
source                  126
make                      0
model                     0
car_availability        620
total_owners              0
broker_quote              0
original_price         3280
car_rating                9
ad_created_on             1
fitness_certificate       8
emi_starts_from           0
booking_down_pymnt        0
reserved                  0
warranty_avail            0
dtype: int64


In [56]:
print(df.shape)

(7400, 29)


In [57]:
print(df.columns)

Index(['car_name', 'yr_mfr', 'fuel_type', 'kms_run', 'sale_price', 'city',
       'times_viewed', 'body_type', 'transmission', 'variant', 'assured_buy',
       'registered_city', 'registered_state', 'is_hot', 'rto', 'source',
       'make', 'model', 'car_availability', 'total_owners', 'broker_quote',
       'original_price', 'car_rating', 'ad_created_on', 'fitness_certificate',
       'emi_starts_from', 'booking_down_pymnt', 'reserved', 'warranty_avail'],
      dtype='object')


In [58]:
df = df[df['sale_price'] > 0].copy()

In [59]:
df=df.drop(columns= [
    'car_name',
    'times_viewed',
    'variant',
    'assured_buy',
    'registered_city',
    'registered_state',
    'is_hot',
    'rto',
    'source',
    'car_availability',
    'broker_quote',
    'original_price',
    'car_rating',
    'ad_created_on',
    'fitness_certificate',
    'emi_starts_from',
    'booking_down_pymnt',
    'reserved',
    'warranty_avail'
])

In [60]:
print(df.shape)

(7397, 10)


In [61]:
print(df.columns)

Index(['yr_mfr', 'fuel_type', 'kms_run', 'sale_price', 'city', 'body_type',
       'transmission', 'make', 'model', 'total_owners'],
      dtype='object')


In [62]:
print(df.head())
print(df.isnull().sum())

   yr_mfr fuel_type  kms_run  sale_price   city  body_type transmission  \
0    2015    petrol     8063      386399  noida  hatchback       manual   
1    2016    petrol    23104      265499  noida  hatchback       manual   
2    2017    petrol    23402      477699  noida  hatchback       manual   
3    2013    diesel    39124      307999  noida  hatchback       manual   
4    2015    petrol    22116      361499  noida  hatchback       manual   

      make      model  total_owners  
0   maruti      swift             2  
1   maruti   alto 800             1  
2  hyundai  grand i10             1  
3   maruti      swift             1  
4  hyundai  grand i10             1  
yr_mfr            0
fuel_type         0
kms_run           0
sale_price        0
city              0
body_type       103
transmission    556
make              0
model             0
total_owners      0
dtype: int64


In [63]:
df['body_type']=df['body_type'].fillna(df['body_type'].mode()[0])
df['transmission']=df['transmission'].fillna(df['transmission'].mode()[0])

In [64]:
x=df.drop('sale_price',axis=1)
y=df['sale_price']
x=pd.get_dummies(x,columns=['fuel_type', 'city', 'body_type', 'transmission', 'make', 'model'],dtype=int)

In [65]:
Q1 = df['sale_price'].quantile(0.25)
Q3 = df['sale_price'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower:", lower)
print("Upper:", upper)

Q1: 281299.0
Q3: 540299.0
IQR: 259000.0
Lower: -107201.0
Upper: 928799.0


In [66]:
outliers = df[
    (df['sale_price'] < lower) |
    (df['sale_price'] > upper)
]

print("Number of outliers:", len(outliers))
print(outliers[['make', 'model', 'yr_mfr', 'kms_run', 'sale_price']].sort_values('sale_price').head(10))
print(outliers[['make', 'model', 'yr_mfr', 'kms_run', 'sale_price']].sort_values('sale_price').tail(10))

Number of outliers: 412
          make          model  yr_mfr  kms_run  sale_price
5479    maruti         ertiga    2017    15074      928999
6309  mahindra        scorpio    2016   142341      929199
3301    maruti  vitara brezza    2018    16351      929799
581    hyundai          creta    2015    59167      930000
5275    maruti           ciaz    2019    43569      930999
4188   hyundai          creta    2016    94775      933299
5263    maruti  vitara brezza    2019    15159      934399
7333   hyundai          creta    2015    86573      934599
6780   renault         duster    2019    14779      936699
1913      ford       ecosport    2019    18721      937399
               make     model  yr_mfr  kms_run  sale_price
4937         toyota  fortuner    2017   188558     2269771
3748         toyota  fortuner    2017    91620     2462277
3350         toyota  fortuner    2017    45112     2558429
7309         toyota  fortuner    2017    81861     2571559
2498         toyota  fortuner   

In [67]:
print(x.head())

   yr_mfr  kms_run  total_owners  fuel_type_diesel  fuel_type_electric  \
0    2015     8063             2                 0                   0   
1    2016    23104             1                 0                   0   
2    2017    23402             1                 0                   0   
3    2013    39124             1                 1                   0   
4    2015    22116             1                 0                   0   

   fuel_type_petrol  fuel_type_petrol & cng  fuel_type_petrol & lpg  \
0                 1                       0                       0   
1                 1                       0                       0   
2                 1                       0                       0   
3                 0                       0                       0   
4                 1                       0                       0   

   city_ahmedabad  city_bengaluru  ...  model_xl6  model_xuv 3oo  \
0               0               0  ...          0           

In [68]:
print(x.shape)
print(x.dtypes.value_counts())
print(x.isnull().sum().sum())

(7397, 240)
int64    240
Name: count, dtype: int64
0


In [69]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [70]:
lr=LinearRegression()
lr.fit(x_train,y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [71]:
y_pre=lr.predict(x_test)
j=pd.DataFrame({"act":y_test,"pre":y_pre})
print(j)

          act           pre
2762   531999  5.740439e+05
2849   232599  1.883003e+05
6686   235299  2.137604e+05
3265   794599  9.622816e+05
486    346499  3.693999e+05
...       ...           ...
6307   300000  2.377830e+05
110    276399  3.234847e+05
51     405399  4.170100e+05
1109   256799  2.135912e+05
3748  2462277  1.510247e+06

[1480 rows x 2 columns]


In [72]:
from sklearn import metrics

mae = metrics.mean_absolute_error(y_test, y_pre)
mse = metrics.mean_squared_error(y_test, y_pre)
rmse = metrics.root_mean_squared_error(y_test, y_pre)
r2 = metrics.r2_score(y_test, y_pre)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R2  :", r2)

MAE : 59090.95032988993
MSE : 12419436989.675016
RMSE: 111442.52774266661
R2  : 0.8430237692980013


In [73]:
print(y.describe())
print(y.mean())

count    7.397000e+03
mean     4.550737e+05
std      2.826111e+05
min      3.500000e+01
25%      2.812990e+05
50%      3.825990e+05
75%      5.402990e+05
max      3.866000e+06
Name: sale_price, dtype: float64
455073.681357307


In [74]:
print((y==0).sum())

0


In [75]:
print(y.sort_values().head(20))

266        35
6277    20000
5625    24000
1632    26000
6001    27000
2546    33000
4453    34000
2556    35000
6865    35000
7394    35000
2610    35000
568     35340
4433    37000
5650    38000
551     39760
2560    40000
2515    41000
2582    41000
2586    45000
6279    45000
Name: sale_price, dtype: int64


In [76]:
j = pd.DataFrame({
    'actual': y_test,
    'predicted': y_pre
})

j['error'] = abs(j['actual'] - j['predicted'])

print(j.sort_values('error', ascending=False).head(10))

       actual     predicted         error
3756  3250000  1.628848e+06  1.621152e+06
2498  2585899  1.545352e+06  1.040547e+06
2861  1917988  8.905933e+05  1.027395e+06
3748  2462277  1.510247e+06  9.520298e+05
2034  1972528  1.106934e+06  8.655942e+05
5492  1830522  1.008512e+06  8.220098e+05
7239   200000  9.935176e+05  7.935176e+05
1210  1538430  9.022556e+05  6.361744e+05
6592   554499  1.138555e+06  5.840558e+05
1565  1566099  1.042282e+06  5.238173e+05
